<a href="https://colab.research.google.com/github/MZiaAfzal71/Average_Weighted_Path_Vector/blob/main/Data%20Files/Models/ChemBERTaModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MZiaAfzal71/Average_Weighted_Path_Vector.git
%cd Average_Weighted_Path_Vector/Data\ Files

In [ ]:
!pip install rdkit

In [3]:
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import List, Optional, Union, Dict, Any
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from tqdm.auto import tqdm

In [4]:
def safe_name(x: str) -> str:
    return str(x).replace(" ", "").replace("/", "_").replace("\\", "_")

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def get_split_indices(df: pd.DataFrame, split_mode="benchmark", test_size=0.2, random_state=42):
    if split_mode == "benchmark":
        train_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "training"].tolist()
        test_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "test"].tolist()

        split_info = df[["SMILES"]].copy()
        split_info["split"] = df["Training/Test"] if "Training/Test" in df.columns else "unknown"
        split_info["split_mode"] = "benchmark"
        return train_idx, test_idx, split_info

    elif split_mode == "scaffold":
        scaffold_df = df[["SMILES"]].copy()
        scaffold_df["scaffold"] = scaffold_df["SMILES"].apply(murcko_scaffold)

        valid_mask = scaffold_df["scaffold"].notna()
        valid_idx = scaffold_df.index[valid_mask]
        groups = scaffold_df.loc[valid_idx, "scaffold"]

        if len(valid_idx) == 0:
            raise ValueError("No valid Murcko scaffolds generated.")

        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_pos, test_pos = next(gss.split(valid_idx, groups=groups))

        train_idx = valid_idx[train_pos].tolist()
        test_idx = valid_idx[test_pos].tolist()

        split_info = scaffold_df.copy()
        split_info["split"] = "unused"
        split_info.loc[train_idx, "split"] = "Training"
        split_info.loc[test_idx, "split"] = "Test"
        split_info["split_mode"] = "scaffold"
        return train_idx, test_idx, split_info

    else:
        raise ValueError(f"Unknown split_mode: {split_mode}")

In [5]:
@dataclass
class Config:
    model_name: str = "seyonec/ChemBERTa-zinc-base-v1"
    output_dir: str = "./chemberta_model_output"
    save_path: str = "best_model.pt"

    max_length: int = 128
    batch_size: int = 16
    epochs: int = 30
    lr: float = 1e-5
    weight_decay: float = 0.01
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dropout: float = 0.1

    split_mode: str = "benchmark"   # "benchmark" or "scaffold"
    test_size: float = 0.2

    train_mode: str = "last2"       # "full", "last2", "head_only"
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    return_numpy: bool = True

# ----------------------------
# Utils
# ----------------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def configure_backbone_trainability(backbone, train_mode="full"):
    # First freeze everything
    for p in backbone.parameters():
        p.requires_grad = False

    if train_mode == "head_only":
        return

    if train_mode == "full":
        for p in backbone.parameters():
            p.requires_grad = True
        return

    # Partial unfreezing: last 2 transformer blocks
    if hasattr(backbone, "encoder") and hasattr(backbone.encoder, "layer"):
        if train_mode == "last2":
            for layer in backbone.encoder.layer[-2:]:
                for p in layer.parameters():
                    p.requires_grad = True

            # also unfreeze embeddings layer norm if present? keep frozen for simplicity
            return

    # fallback if architecture differs
    if train_mode != "head_only":
        for p in backbone.parameters():
            p.requires_grad = True

class ChemBERTaModel(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1, train_mode: str = "full"):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        H = self.bert.config.hidden_size

        configure_backbone_trainability(self.bert, train_mode=train_mode)

        self.main = nn.Sequential(
            nn.Linear(H, H // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H // 2, 1),
        )

    def forward(self, input_ids, attention_mask, targets=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        y_pred = self.main(cls).squeeze(-1)

        loss = None
        if targets is not None:
            loss = F.mse_loss(y_pred, targets.float())
        return y_pred, loss

# ----------------------------
# Dataset / Collate
# ----------------------------
class SmileDataset(Dataset):
    def __init__(self, smiles: List[str], targets: Optional[np.ndarray],
                 tokenizer: AutoTokenizer, max_length: int):
        self.smiles = list(smiles)
        self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.smiles)

    def __getitem__(self, i):
        enc = self.tok(self.smiles[i],
                       truncation=True, padding="max_length",
                       max_length=self.max_length, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.targets is not None:
            item["labels"] = torch.tensor(self.targets[i], dtype=torch.float32)
        return item

def collate_stack(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in batch[0] if k != "labels"}
    if "labels" in batch[0]:
        out["labels"] = torch.stack([b["labels"] for b in batch])
    return out

def make_smiles_loaders(df: pd.DataFrame, target_col: str, tokenizer: AutoTokenizer, cfg: Config):
    train_idx, test_idx, split_info = get_split_indices(
        df,
        split_mode=cfg.split_mode,
        test_size=cfg.test_size,
        random_state=cfg.seed
    )

    train_df = df.loc[train_idx].reset_index(drop=True).copy()
    test_df = df.loc[test_idx].reset_index(drop=True).copy()

    train_ds = SmileDataset(
        train_df["SMILES"].tolist(),
        train_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length
    )
    test_ds = SmileDataset(
        test_df["SMILES"].tolist(),
        test_df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length
    )

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_stack)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)

    return train_loader, test_loader, train_df, test_df, split_info

def setup_optimizer_scheduler(model, train_dataloader, epochs, lr=2e-5, warmup_ratio=0.1):

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr = lr, weight_decay=0.01)

    total_steps = len(train_dataloader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    return optimizer, scheduler

def train_model(model, train_loader, val_loader, optimizer, scheduler,
                device, epochs=10, grad_clip=1.0, save_path="best_model.pt"):
    """
    Full trainer loop for ChemBERTaModel.

    Args:
      model: nn.Module
      train_loader: DataLoader
      val_loader: DataLoader
      optimizer: torch.optim.Optimizer
      scheduler: torch.optim.lr_scheduler
      device: torch.device
      epochs: int
      grad_clip: float (gradient clipping norm)
      save_path: str (where to save best model)
    """
    model.to(device)
    best_val = float("inf")

    for epoch in range(1, epochs+1):
        # ---- TRAIN ----
        model.train()
        train_loss, n_train = 0.0, 0
        diag_accum = {}

        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
        for batch in pbar:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["labels"].to(device)

            preds, loss= model(input_ids, attention_mask, targets)

            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()

            bs = input_ids.size(0)
            train_loss += loss.item() * bs
            n_train += bs

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss /= n_train
        diag_accum = {k: v / n_train for k, v in diag_accum.items()}

        # ---- VALIDATION ----
        model.eval()
        val_loss, n_val = 0.0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["labels"].to(device)

                preds, loss = model(input_ids, attention_mask, targets)

                bs = input_ids.size(0)
                val_loss += loss.item() * bs
                n_val += bs

        val_loss /= n_val

        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, ")

        # ---- Save best ----
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved best model (val_loss={val_loss:.4f})")

    print("Training complete.")
    model.load_state_dict(torch.load(save_path))
    return model

def predict(model, test_loader, device, return_numpy=True):
    """
    Run inference on a test set.

    Args:
      model: trained ChemBERTaFusionV2
      test_loader: DataLoader
      device: torch.device
      return_numpy: if True, returns numpy array

    Returns:
      preds: [N] predictions (torch.Tensor or np.ndarray)
    """
    model.eval()
    model.to(device)
    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            preds, _ = model(input_ids, attention_mask)
            all_preds.append(preds.cpu())

    preds = torch.cat(all_preds, dim=0)

    if return_numpy:
        return preds.numpy()
    return preds

def train_for_prop(file: str, prop: str, cfg: Config) -> Dict[str, Any]:
    set_seed(cfg.seed)
    ensure_dir(cfg.output_dir)

    target_col = f"{prop}-Measured"
    safe_prop = safe_name(prop)
    run_name = f"{safe_prop}_{cfg.split_mode}_{cfg.train_mode}"

    try:
        df = pd.read_excel(file, sheet_name=prop)
    except Exception as e:
        raise ValueError(f"Failed to read sheet '{prop}' from {file}: {repr(e)}")

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

    train_loader, test_loader, train_df, test_df, split_info = make_smiles_loaders(
        df, target_col, tokenizer, cfg
    )

    model = ChemBERTaModel(
        cfg.model_name,
        dropout=cfg.dropout,
        train_mode=cfg.train_mode
    ).to(cfg.device)

    optimizer, scheduler = setup_optimizer_scheduler(
        model, train_loader, cfg.epochs, cfg.lr, cfg.warmup_ratio
    )

    save_model = os.path.join(cfg.output_dir, f"{run_name}_{cfg.save_path}")
    model = train_model(
        model,
        train_loader,
        test_loader,
        optimizer,
        scheduler,
        cfg.device,
        cfg.epochs,
        cfg.grad_clip,
        save_model
    )

    # test predictions
    test_preds = predict(model, test_loader, cfg.device, cfg.return_numpy)
    y_test = test_df[target_col].to_numpy(dtype=np.float32)

    mae_v = mean_absolute_error(y_test, test_preds)
    rmse_v = rmse(y_test, test_preds)
    r2_v = r2_score(y_test, test_preds)
    print(f"Final Test metrics for {run_name} → MAE: {mae_v:.4f} | RMSE: {rmse_v:.4f} | R²: {r2_v:.4f}")

    split_path = os.path.join(cfg.output_dir, f"{run_name}_split.csv")
    split_info.to_csv(split_path, index=False)

    test_results = test_df.copy()
    test_results[f"{prop} Prediction"] = test_preds
    test_pred_path = os.path.join(cfg.output_dir, f"{run_name}_test_predictions.xlsx")
    test_results.to_excel(test_pred_path, index=False)

    # optional all-row predictions
    all_ds = SmileDataset(
        df["SMILES"].tolist(),
        df[target_col].to_numpy(dtype=np.float32),
        tokenizer,
        cfg.max_length
    )
    all_loader = DataLoader(all_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)
    all_preds = predict(model, all_loader, cfg.device, cfg.return_numpy)

    all_results = df.copy()
    all_results[f"{prop} Prediction"] = all_preds
    all_pred_path = os.path.join(cfg.output_dir, f"{run_name}_all_predictions.xlsx")
    all_results.to_excel(all_pred_path, index=False)

    return {
        "Property": prop,
        "split_mode": cfg.split_mode,
        "train_mode": cfg.train_mode,
        "MAE": mae_v,
        "RMSE": rmse_v,
        "R2": r2_v,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "best_path": save_model
    }

In [ ]:
cfg = Config(
    output_dir="chemberta_results",
    epochs=30,
    batch_size=8,
    max_length=128,
    train_mode="last2",      # "full", "last2", "head_only"
    split_mode="benchmark",  # or "scaffold"
    lr=1e-5,
)

file = "Excel Files/Zang_Data.xlsx"
prop_names = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]

perf_rows = []
for prop in prop_names:
    res = train_for_prop(file, prop, cfg)
    perf_rows.append(res)

perf_df = pd.DataFrame(perf_rows)
perf_df.to_csv(os.path.join(cfg.output_dir, f"chemberta_{cfg.split_mode}_{cfg.train_mode}_stats.csv"), index=False)
perf_df